In [ ]:
import kagglehub
import os
import numpy as np
import tensorflow as tf
from google.colab import drive
drive.mount('/content/drive')

from tensorflow.keras import mixed_precision

mixed_precision.set_global_policy("mixed_float16")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
path = kagglehub.dataset_download(
    "aresusayhi/vsl-vietnamese-sign-languages"
)

print(path)

Using Colab cache for faster access to the 'vsl-vietnamese-sign-languages' dataset.
/kaggle/input/vsl-vietnamese-sign-languages


In [ ]:
train_path = f"{path}/Processed/train"
val_path = f"{path}/Processed/val"
test_path = f"{path}/Processed/test"

labels = sorted(os.listdir(train_path))
NUM_CLASSES = len(labels)

train_files = []
val_files = []
test_files = []

for cls in labels:

    for f in os.listdir(os.path.join(train_path, cls)):
        train_files.append(
            os.path.join(train_path, cls, f)
        )

    for f in os.listdir(os.path.join(val_path, cls)):
        val_files.append(
            os.path.join(val_path, cls, f)
        )

    for f in os.listdir(os.path.join(test_path, cls)):
        test_files.append(
            os.path.join(test_path, cls, f)
        )

print(len(train_files))

145019


In [ ]:
def generator(file_list):

    for file in file_list:

        data = np.load(file)

        x = data["sequence"].astype(np.float32)

        y = int(data["label"])

        yield x, y

In [ ]:
train_ds = tf.data.Dataset.from_generator(
    lambda: generator(train_files),
    output_signature=(
        tf.TensorSpec(
            shape=(60,201),
            dtype=tf.float32
        ),
        tf.TensorSpec(
            shape=(),
            dtype=tf.int32
        )
    )
)

val_ds = tf.data.Dataset.from_generator(
    lambda: generator(val_files),
    output_signature=(
        tf.TensorSpec((60,201), tf.float32),
        tf.TensorSpec((), tf.int32)
    )
)

test_ds = tf.data.Dataset.from_generator(
    lambda: generator(test_files),
    output_signature=(
        tf.TensorSpec((60,201), tf.float32),
        tf.TensorSpec((), tf.int32)
    )
)

In [ ]:
BATCH_SIZE = 128

train_ds = train_ds.shuffle(10000)

train_ds = train_ds.batch(BATCH_SIZE)

val_ds = val_ds.batch(BATCH_SIZE)

test_ds = test_ds.batch(BATCH_SIZE)

train_ds = train_ds.prefetch(
    tf.data.AUTOTUNE
)

val_ds = val_ds.prefetch(
    tf.data.AUTOTUNE
)

test_ds = test_ds.prefetch(
    tf.data.AUTOTUNE
)

In [ ]:
from tensorflow.keras.layers import Layer
import tensorflow as tf

class Attention(Layer):

    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def build(self, input_shape):

        self.W = self.add_weight(
            name="attention_weight",
            shape=(input_shape[-1], 1),
            initializer="glorot_uniform",
            trainable=True
        )

        self.b = self.add_weight(
            name="attention_bias",
            shape=(input_shape[1], 1),
            initializer="zeros",
            trainable=True
        )

        super().build(input_shape)

    def call(self, x):

        score = tf.nn.tanh(
            tf.matmul(x, self.W) + self.b
        )

        weights = tf.nn.softmax(
            score,
            axis=1
        )

        context = weights * x

        context = tf.reduce_sum(
            context,
            axis=1
        )

        return context

In [ ]:
from tensorflow.keras.layers import (
    Input,
    Masking,
    Bidirectional,
    LSTM,
    Dense,
    Dropout,
    BatchNormalization
)

from tensorflow.keras.models import Model

inputs = Input(shape=(60, 201))

x = Masking(mask_value=0.0)(inputs)

x = Bidirectional(
    LSTM(
        256,
        return_sequences=True
    )
)(x)

x = BatchNormalization()(x)
x = Dropout(0.3)(x)

x = Bidirectional(
    LSTM(
        128,
        return_sequences=True
    )
)(x)

x = BatchNormalization()(x)
x = Dropout(0.3)(x)

x = Attention()(x)

x = Dense(
    512,
    activation="relu"
)(x)

x = BatchNormalization()(x)
x = Dropout(0.4)(x)

outputs = Dense(
    NUM_CLASSES,
    activation="softmax",
    dtype="float32"
)(x)

model = Model(
    inputs,
    outputs
)

model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-3
    ),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"]
)

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/layer.py:982: UserWarning: Layer 'attention' (of type Attention) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 60, 201)   │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cast_1 (Cast)       │ (None, 60, 201)   │          0 │ input_layer_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_1         │ (None, 60, 201)   │          0 │ cast_1[0][0]      │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ masking_1 (Masking) │ (None, 60, 201)   │          0 │ input_layer_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ any_1 (Any)         │ (None, 60)        │          0 │ not_equal_1[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_2     │ (None, 60, 512)   │    937,984 │ masking_1[0][0],  │
│ (Bidirectional)     │                   │            │ any_1[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 60, 512)   │      2,048 │ bidirectional_2[… │
│ (BatchNormalizatio… │                   │            │ any_1[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 60, 512)   │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_3     │ (None, 60, 256)   │    656,384 │ dropout_2[0][0],  │
│ (Bidirectional)     │                   │            │ any_1[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 60, 256)   │      1,024 │ bidirectional_3[… │
│ (BatchNormalizatio… │                   │            │ any_1[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, 60, 256)   │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attention           │ (None, 256)       │        316 │ dropout_3[0][0]   │
│ (Attention)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 512)       │    131,584 │ attention[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 512)       │      2,048 │ dense[0][0]       │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_4 (Dropout) │ (None, 512)       │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 3315)      │  1,700,595 │ dropout_4[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 3,431,983 (13.09 MB)

 Trainable params: 3,429,423 (13.08 MB)

 Non-trainable params: 2,560 (10.00 KB)

In [ ]:
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau,
    ModelCheckpoint
)

# callbacks = [

#     EarlyStopping(
#         monitor="val_accuracy",
#         patience=5,
#         restore_best_weights=True
#     ),

#     ReduceLROnPlateau(
#         monitor="val_loss",
#         factor=0.5,
#         patience=2,
#         verbose=1
#     ),

#     ModelCheckpoint(
#         "best_model.keras",
#         monitor="val_accuracy",
#         save_best_only=True,
#         verbose=1
#     )
# ]
from tensorflow.keras.callbacks import (
    ModelCheckpoint,
    EarlyStopping,
    ReduceLROnPlateau,
    CSVLogger
)

callbacks = [

    ModelCheckpoint(
        "/content/drive/MyDrive/VSL/best_model.keras",
        monitor="val_accuracy",
        save_best_only=True,
        verbose=1
    ),

    ModelCheckpoint(
        "/content/drive/MyDrive/VSL/epoch_{epoch:02d}.keras",
        save_freq="epoch",
        verbose=1
    ),

    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        verbose=1
    ),

    EarlyStopping(
        monitor="val_accuracy",
        patience=5,
        restore_best_weights=True
    ),

    CSVLogger(
        "/content/drive/MyDrive/VSL/training_log.csv",
        append=True
    )
]

In [ ]:
import os

os.makedirs("/content/drive/MyDrive/VSL", exist_ok=True)
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=30,
    callbacks=callbacks
)

In [ ]:
loss, acc = model.evaluate(test_ds)

print(f"\nTest Accuracy: {acc*100:.2f}%")

In [ ]:
!pip install mediapipe opencv-python

In [ ]:
import cv2
import mediapipe as mp
import numpy as np
import tensorflow as tf

In [ ]:
model = tf.keras.models.load_model(
    "/content/drive/MyDrive/VSL/epoch_09.keras",
    custom_objects={
        "Attention": Attention
    }
)

print("Model loaded!")



labels = sorted(os.listdir(train_path))

idx_to_label = {

    i: labels[i]

    for i in range(len(labels))

}

Model loaded!


In [ ]:
!wget -q https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_lite/float16/latest/pose_landmarker_lite.task

In [ ]:
import mediapipe as mp

from mediapipe.tasks import python

from mediapipe.tasks.python import vision


base_options = python.BaseOptions(

    model_asset_path="pose_landmarker_lite.task"

)

options = vision.PoseLandmarkerOptions(

    base_options=base_options,

    running_mode=vision.RunningMode.VIDEO

)

# detector = vision.PoseLandmarker.create_from_options(

#     options

# )

print("Detector configuration loaded")

Detector configuration loaded


In [ ]:
import numpy as np


def extract_201(result):
    feats = []
    if len(result.pose_landmarks) > 0:
        pose = result.pose_landmarks[0]
        for lm in pose:
            feats.extend([
                lm.x,
                lm.y,
                lm.z
            ])

    feats = np.array(
        feats,
        dtype=np.float32
    )

    # Pose có 99 số

    # Pad lên 201

    if len(feats) < 201:
        feats = np.pad(
            feats,
            (
                0,
                201-len(feats)
            )
        )

    else:
        feats = feats[:201]
    return feats

In [ ]:
# import cv2
# import mediapipe as mp


# def video_to_sequence(video_path):
#     # Create a new detector instance for each video
#     detector = mp.tasks.vision.PoseLandmarker.create_from_options(options)

#     cap = cv2.VideoCapture(
#         video_path
#     )

#     total = int(
#         cap.get(
#             cv2.CAP_PROP_FRAME_COUNT
#         )
#     )
#     idx = np.linspace(
#         0,
#         total-1,
#         60
#     ).astype(int)
#     sequence = []
#     frame_id = 0
#     pick = 0
#     while True:
#         ret, frame = cap.read()
#         if not ret:
#             break
#         if pick < 60 and frame_id == idx[pick]:
#             rgb = cv2.cvtColor(
#                 frame,
#                 cv2.COLOR_BGR2RGB
#             )
#             mp_image = mp.Image(
#                 image_format=mp.ImageFormat.SRGB,
#                 data=rgb
#             )

#             result = detector.detect_for_video(
#                 mp_image,
#               timestamp_ms=frame_id*33
#             )

#             feat = extract_201(
#                 result
#             )

#             sequence.append(
#                 feat
#             )

#             pick += 1
#         frame_id += 1
#     cap.release()
#     # Close the detector after processing the video
#     detector.close()
#     sequence = np.array(
#         sequence,
#         dtype=np.float32
#     )

#     if len(sequence) < 60:
#         pad = np.zeros((
#                 60-len(sequence),
#                 201
#             ),

#             dtype=np.float32 )

#         sequence = np.concatenate(
#             [
#                 sequence,
#                 pad
#             ],
#             axis=0 )
#     return sequence

In [ ]:
import cv2

def video_to_sequence(video_path):
    cap = cv2.VideoCapture(
        video_path
    )
    total = int(
        cap.get(
            cv2.CAP_PROP_FRAME_COUNT
        )
    )
    idx = np.linspace(
        0,
        total-1,
        60

    ).astype(int)
    sequence = []
    frame_id = 0
    pick = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if pick < 60 and frame_id == idx[pick]:
            rgb = cv2.cvtColor(
                frame,
                cv2.COLOR_BGR2RGB
            )

            mp_image = mp.Image(
                image_format=mp.ImageFormat.SRGB,
                data=rgb
            )

            result = detector.detect_for_video(
                mp_image,
                timestamp_ms=frame_id*33
            )

            feat = extract_201(
                result
            )

            sequence.append(
                feat
            )
            pick += 1
        frame_id += 1
    cap.release()
    sequence = np.array(
        sequence,
        dtype=np.float32

    )

    if len(sequence) < 60:
        pad = np.zeros(
            (
                60-len(sequence),
                201
            ),
            dtype=np.float32
        )
        sequence = np.concatenate(
            [
                sequence,
                pad
            ],
            axis=0
        )
    return sequence

In [ ]:
def predict_video(video_path):
    seq = video_to_sequence(
        video_path)
    print(
        seq.shape )

    x = np.expand_dims(
        seq,
        axis=0)

    pred = model.predict(
        x,
        verbose=0 )[0]
    top5 = np.argsort(
        pred
    )[::-1][:5]
    print()
    print("Top 5:")
    print()
    for i in top5:
        print(
            idx_to_label[i],
            float(pred[i]))

    cls = np.argmax(
        pred)

    print()
    print(
        "Prediction:")

    print(
        idx_to_label[cls])

    return idx_to_label[cls]

In [ ]:
video_path = f"{path}/Dataset/Videos/D0190B.mp4"

predict_video(video_path)

ValueError: Input timestamp must be monotonically increasing.

In [ ]:
import os
import random
import numpy as np
import mediapipe as mp

from mediapipe.tasks import python
from mediapipe.tasks.python import vision


# Tạo detector mới mỗi lần
def create_detector():

    base_options = python.BaseOptions(
        model_asset_path="pose_landmarker_lite.task"
    )
    options = vision.PoseLandmarkerOptions(
        base_options=base_options,
        running_mode=vision.RunningMode.VIDEO
    )

    return vision.PoseLandmarker.create_from_options(
        options
    )

video_dir = f"{path}/Dataset/Videos"
videos = [
    v for v in os.listdir(video_dir)
    if v.endswith(".mp4")
]

samples = random.sample(
    videos,
    min(50,len(videos))

)


success = 0
failed = 0
for i,v in enumerate(samples):
    print("\n"+"="*60)
    print(f"[{i+1}/50] {v}")
    try:

        detector = create_detector()
        cap = cv2.VideoCapture(
            os.path.join(
                video_dir,

                v
            )
        )

        total = int(
            cap.get(
                cv2.CAP_PROP_FRAME_COUNT
            )
        )

        idx = np.linspace(
            0,
            max(total-1,0),
            60
        ).astype(int)
        sequence = []
        frame_id = 0
        pick = 0

        while True:
            ret, frame = cap.read()
            if not ret:
                break
            if (
                pick < len(idx)
                and
                frame_id == idx[pick]
            ):
                rgb = cv2.cvtColor(
                    frame,
                    cv2.COLOR_BGR2RGB
                )
                mp_image = mp.Image(
                    image_format=mp.ImageFormat.SRGB,
                    data=rgb

                )

                result = detector.detect_for_video(
                    mp_image,
                    timestamp_ms=int(
                        frame_id*1000/30
                    )
                )
                feats=[]
                if len(result.pose_landmarks)>0:
                    pose = result.pose_landmarks[0]
                    for lm in pose:
                        feats.extend([
                            lm.x,
                            lm.y,
                            lm.z
                        ])

                feats=np.array(
                    feats,
                    dtype=np.float32
                )
                if len(feats)<201:
                    feats=np.pad(
                        feats,
                        (
                            0,
                            201-len(feats) ) )
                else:
                    feats=feats[:201]
                    sequence.append(
                    feats
                )
                pick+=1
            frame_id+=1
        cap.release()
        sequence=np.array(
            sequence,
            dtype=np.float32
        )
        if len(sequence)<60:
            pad=np.zeros(
                (
                    60-len(sequence),
                    201
                ),
                dtype=np.float32
            )
            sequence=np.concatenate(
                [
                    sequence,
                    pad
                ],
                axis=0
            )
        x=np.expand_dims(
            sequence,
            axis=0
        )
        pred=model.predict(
            x,
            verbose=0
        )[0]
        cls=np.argmax(pred)
        conf=float(pred[cls])
        print(
            "Prediction:",
            idx_to_label[cls]
        )
        print(
            "Confidence:",
            f"{conf:.2%}"
        )
        success+=1
    except Exception as e:
        print(
            "ERROR:",
            e
        )

        failed+=1

print("\n"+"="*60)
print("Success:",success)
print("Failed:",failed)


[1/50] W03810.mp4
Prediction: lưỡi câu
Confidence: 67.44%

[2/50] W02305.mp4
Prediction: bông hoa
Confidence: 89.43%

[3/50] W00822.mp4
Prediction: bông hoa
Confidence: 53.22%

[4/50] W00255T.mp4
Prediction: bông hoa
Confidence: 99.38%

[5/50] W00522.mp4
Prediction: lưỡi câu
Confidence: 40.52%

[6/50] W03649T.mp4
Prediction: bánh giò
Confidence: 29.88%

[7/50] W01898N.mp4
Prediction: con chuồn chuồn
Confidence: 76.03%

[8/50] W01755.mp4
Prediction: bông hoa
Confidence: 49.36%

[9/50] W02264N.mp4
Prediction: lưỡi câu
Confidence: 71.20%

[10/50] W03314B.mp4
Prediction: bông hoa
Confidence: 95.12%

[11/50] W00579.mp4
Prediction: bông hoa
Confidence: 43.15%

[12/50] W03650.mp4
Prediction: hấp
Confidence: 60.09%

[13/50] W02570.mp4
Prediction: bông hoa
Confidence: 95.28%

[14/50] W04070.mp4
Prediction: màu xanh lá cây
Confidence: 67.99%

[15/50] W00329.mp4
Prediction: đôi tay
Confidence: 27.06%

[16/50] W01479.mp4
Prediction: màu xanh lá cây
Confidence: 95.52%

[17/50] W01849.mp4
Predictio